# 🚀 Viettel AI Race 2026 — LFM2.5 Colab Fix
**Sửa triệt để lỗi `libcudart.so.13` trên Colab** bằng cách link CUDA 12 runtime thư viện chuẩn.

## Bước 1: Fix CUDA 13 symlink & Cài đặt vLLM

In [ ]:
# 1. Sửa lỗi libcudart.so.13 trên Colab (Tạo symlink từ CUDA 12 sẵn có của Colab)
!apt-get update -qq && apt-get install -y -qq libcudart12 2>/dev/null || true
!find /usr/local/cuda* /usr/lib -name "libcudart.so*" -exec ln -sf {} /usr/local/lib/libcudart.so.13 \; 2>/dev/null || true
!ldconfig /usr/local/lib

# 2. Gỡ bỏ các gói xung đột ABI trên Colab
!pip uninstall -y torchaudio torchvision vllm
!rm -rf /usr/local/lib/python*/dist-packages/torchaudio* /usr/local/lib/python*/dist-packages/torchvision*

# 3. Cài đặt vLLM chuẩn CUDA 12
!pip install -q --extra-index-url https://download.pytorch.org/whl/cu121 vllm aiohttp openai numpy huggingface_hub

import torch
print(f"\nPyTorch CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("❌ Không có GPU! Vào Runtime → Change runtime type → chọn T4 GPU")

## Bước 2: Tải Model

In [ ]:
import os
from huggingface_hub import snapshot_download

model_dir = "./model"
if not os.path.exists(model_dir):
    print("Downloading LiquidAI/LFM2.5-1.2B-Instruct...")
    snapshot_download(repo_id="LiquidAI/LFM2.5-1.2B-Instruct", local_dir=model_dir)
    print("Done!")
else:
    print("Model already at ./model")

## Bước 3: Khởi chạy vLLM Server

In [ ]:
import subprocess, time, requests, os, sys

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = "/usr/local/lib:" + env.get("LD_LIBRARY_PATH", "")

vllm_cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model=./model",
    "--served-model-name=LFM2.5-1.2B-Instruct",
    "--host=0.0.0.0",
    "--port=8000",
    "--tensor-parallel-size=1",
    "--max-model-len=4096",
    "--gpu-memory-utilization=0.85",
    "--enable-prefix-caching",
    "--trust-remote-code",
]

print("Command:", ' '.join(vllm_cmd))
!pkill -f "vllm.entrypoints.openai.api_server"
time.sleep(2)

log_file = open("vllm_colab.log", "w")
server_process = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=log_file, env=env)
print("\nServer starting... (chờ tối đa 120s)")

ready = False
for i in range(60):
    try:
        resp = requests.get("http://localhost:8000/health", timeout=2)
        if resp.status_code == 200:
            print(f"\n✅ Server sẵn sàng sau {(i+1)*2}s!")
            ready = True
            break
    except:
        pass
    if (i+1) % 5 == 0:
        print(f"  Đang chờ... {(i+1)*2}s")
    time.sleep(2)

if not ready:
    print("\n❌ SERVER KHÔNG KHỞI ĐỘNG ĐƯỢC!")
    log_file.close()
    with open("vllm_colab.log", "r") as f:
        print(f.read()[-2000:])
else:
    print("✅ PASS: Server vLLM khởi động OK!")